# 06 — Combined Strategy: Momentum + Mean Reversion

**Universe:** 14-asset multi-asset portfolio (11 sector ETFs + TLT + GLD + IEF)  
**Period:** 2018-01-01 to 2024-12-31  
**Benchmark:** SPY Buy & Hold

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import plotly.express as px

from backtest_engine.data.loader import load_prices
from backtest_engine.strategies.multi_asset import UNIVERSE, risk_parity_weights
from backtest_engine.strategies.momentum import momentum_signals
from backtest_engine.strategies.mean_reversion import mean_reversion_signals
from backtest_engine.strategies.portfolio import blend_signals
from backtest_engine.strategies.regime import trend_filter, apply_regime_filter
from backtest_engine.backtest.engine import run_backtest
from backtest_engine.metrics.performance import sharpe_ratio, cagr, max_drawdown, calmar_ratio, win_rate

START, END = '2018-01-01', '2024-12-31'
prices_all = load_prices(UNIVERSE + ['SPY'], START, END, cache=True)
spy    = prices_all['SPY']
prices = prices_all[UNIVERSE]
print(f'Loaded {len(prices)} rows x {len(prices.columns)} tickers')

Loaded 1644 rows x 14 tickers


## 1. Build signals

In [2]:
regime = trend_filter(spy, sma_window=200)
rets   = prices.pct_change()

# Momentum: binary signals, no shift -- engine handles shift(1)
mom_sig = apply_regime_filter(
    momentum_signals(prices, lookback_months=12, skip_months=1, top_n=3), regime
)

# Mean reversion: binary signals, no shift -- consistent with momentum
mr_sig = mean_reversion_signals(
    prices, rsi_period=5, rsi_oversold=35.0, rsi_exit=50.0,
    zscore_window=20, zscore_threshold=-2.0, max_hold_days=5, regime=regime,
)

print(f'Momentum active days   : {(mom_sig.sum(axis=1) > 0).sum()}')
print(f'Mean-reversion act days: {(mr_sig.sum(axis=1) > 0).sum()}')

Momentum active days   : 1106
Mean-reversion act days: 409


## 2. Backtests

In [3]:
COST_BPS = 10
CAPITAL  = 100_000.0

# Individual strategies via engine (computes risk-parity weights + shift internally)
res_mom = run_backtest(prices, mom_sig, initial_capital=CAPITAL,
                       transaction_cost_bps=COST_BPS, slippage_bps=0,
                       sizing_method='risk_parity')
res_mr  = run_backtest(prices, mr_sig,  initial_capital=CAPITAL,
                       transaction_cost_bps=COST_BPS, slippage_bps=0,
                       sizing_method='risk_parity')

# Blended: compute unshifted risk-parity weights, blend, apply shift once, manual equity
blend_w_exec = blend_signals(
    {'mom': (risk_parity_weights(mom_sig, rets, lookback_days=60), 0.6),
     'mr':  (risk_parity_weights(mr_sig,  rets, lookback_days=60), 0.4)}
).shift(1).fillna(0.0)

def _bt(prices, w, cost_bps=10, capital=100_000.0):
    log_r = np.log(prices / prices.shift(1))
    gross = (w * log_r).sum(axis=1, skipna=True)
    delta = (w - w.shift(1).fillna(0.0)).abs().sum(axis=1)
    net   = gross - delta * cost_bps / 10_000
    eq    = pd.Series(capital * np.exp(net.cumsum().to_numpy()), index=prices.index)
    return eq, eq.pct_change().fillna(0.0)

blend_eq, blend_ret = _bt(prices, blend_w_exec, cost_bps=COST_BPS, capital=CAPITAL)
spy_eq  = spy / spy.iloc[0] * CAPITAL
spy_ret = spy.pct_change().dropna()
print('Backtests complete.')

Backtests complete.


## 3. Equity curves

In [4]:
eq = pd.DataFrame({'Momentum': res_mom.equity_curve, 'Mean Reversion': res_mr.equity_curve,
                   '60/40 Blend': blend_eq, 'SPY B&H': spy_eq.reindex(blend_eq.index)})
px.line(eq, title='Equity Curves', labels={'value': 'Portfolio Value ($)', 'variable': 'Strategy'}).show()

## 4. Performance summary

In [5]:
def _row(returns, label):
    r = returns.dropna()
    return {'Strategy': label, 'CAGR': f'{cagr(r):.2%}', 'Sharpe': f'{sharpe_ratio(r):.3f}',
            'Max DD': f'{max_drawdown(r):.2%}', 'Calmar': f'{calmar_ratio(r):.3f}',
            'Win Rate': f'{win_rate(r):.1%}'}

spy_yrs  = (spy.index[-1] - spy.index[0]).days / 365.25
spy_cagr = (spy.iloc[-1] / spy.iloc[0]) ** (1 / spy_yrs) - 1
spy_sh   = spy_ret.mean() / spy_ret.std() * 252**0.5
spy_dd   = ((spy - spy.cummax()) / spy.cummax()).min()

print(pd.DataFrame([
    _row(res_mom.returns, 'Momentum (12-1, top3)'),
    _row(res_mr.returns,  'Mean Reversion (RSI5)'),
    _row(blend_ret,       '60/40 Blend'),
    {'Strategy': 'SPY B&H', 'CAGR': f'{spy_cagr:.2%}', 'Sharpe': f'{spy_sh:.3f}',
     'Max DD': f'{spy_dd:.2%}', 'Calmar': f'{spy_cagr/abs(spy_dd):.3f}', 'Win Rate': '---'},
]).to_string(index=False))

             Strategy   CAGR Sharpe  Max DD Calmar Win Rate
Momentum (12-1, top3)  6.54%  0.599 -17.53%  0.373    36.3%
Mean Reversion (RSI5) -1.34% -0.116 -23.99% -0.056    12.6%
          60/40 Blend  3.34%  0.412 -17.09%  0.196    37.3%
              SPY B&H 14.13%  0.773 -33.72%  0.419      ---


## 5. Correlation matrix

In [6]:
corr_df = pd.DataFrame({'Momentum': res_mom.returns, 'Mean Reversion': res_mr.returns,
                         '60/40 Blend': blend_ret,
                         'SPY': spy_ret.reindex(blend_ret.index).fillna(0)}).corr()
px.imshow(corr_df, color_continuous_scale='RdYlGn', color_continuous_midpoint=0,
          text_auto='.2f', title='Return Correlation Matrix').show()
c = corr_df.loc['Momentum', 'Mean Reversion']
print(f'Momentum vs MR corr: {c:.3f} -- {"Low (complementary)" if c < 0.3 else "Moderate"}')

Momentum vs MR corr: 0.396 -- Moderate


## 6. 2022 Stress Test

In [7]:
def _yr(s, y):
    sl = s.loc[f'{y}-01-01':f'{y}-12-31']
    return sl.iloc[-1] / sl.iloc[0] - 1 if len(sl) > 1 else float('nan')
print('2022 returns:')
for name, s in [('Momentum', res_mom.equity_curve), ('Mean Reversion', res_mr.equity_curve),
                ('60/40 Blend', blend_eq), ('SPY B&H', spy_eq)]:
    print(f'  {name:<22}: {_yr(s, 2022):+.2%}')
eq_2022 = eq.loc['2022-01-01':'2022-12-31']
px.line(eq_2022 / eq_2022.iloc[0], title='2022 Stress Test').show()

2022 returns:
  Momentum              : -5.19%
  Mean Reversion        : -2.47%
  60/40 Blend           : -4.11%
  SPY B&H               : -18.65%


## 7. Walk-Forward Validation

In [8]:
from backtest_engine.backtest.walk_forward import walk_forward_test

# IMPORTANT: pass only prices[UNIVERSE] (14 cols) to walk_forward_test.
# walk_forward calls run_backtest(test_prices, strategy_fn_output) — shapes must match.
# SPY is captured from the outer scope for the regime filter.
def blend_fn(prices_slice: pd.DataFrame, **p) -> pd.DataFrame:
    lb  = int(p.get('lookback_months', 12))
    ros = float(p.get('rsi_oversold', 35.0))
    bm  = float(p.get('blend_momentum', 0.6))
    spy_aligned = spy.reindex(prices_slice.index).ffill()  # spy from outer scope
    r    = trend_filter(spy_aligned, sma_window=200)
    ret_ = prices_slice.pct_change()
    ms   = apply_regime_filter(
        momentum_signals(prices_slice, lookback_months=lb, skip_months=1, top_n=3), r
    )
    mrs  = mean_reversion_signals(prices_slice, rsi_period=5, rsi_oversold=ros, rsi_exit=50.0,
                                   zscore_window=20, zscore_threshold=-2.0, max_hold_days=5, regime=r)
    return blend_signals({'mom': (risk_parity_weights(ms, ret_, 60), bm),
                          'mr':  (risk_parity_weights(mrs, ret_, 60), 1.0 - bm)})

print('Running walk-forward (3y train / 1y test) ...')
wf = walk_forward_test(
    prices,  # 14-col UNIVERSE prices only
    blend_fn,
    param_grid={'lookback_months': [9, 12, 15], 'rsi_oversold': [30.0, 35.0, 40.0], 'blend_momentum': [0.5, 0.6, 0.7]},
    train_years=3, test_years=1, metric='sharpe_ratio',
)
print('Done.')

Running walk-forward (3y train / 1y test) ...


Done.


In [9]:
print(f'Avg IS Sharpe : {wf.avg_is_metric:.3f}')
print(f'Avg OOS Sharpe: {wf.oos_metric:.3f}')
if wf.avg_is_metric != 0:
    print(f'IS->OOS degradation: {(1 - wf.oos_metric / wf.avg_is_metric):.1%}')
for i, w in enumerate(wf.windows):
    print(f'  W{i+1} {w.test_start.date()}->{w.test_end.date()}  best={w.best_params}  IS={w.is_metric:.3f}  OOS={w.oos_metric:.3f}')

Avg IS Sharpe : 0.947
Avg OOS Sharpe: -0.019
IS->OOS degradation: 102.0%
  W1 2021-06-19->2022-06-19  best={'lookback_months': 9, 'rsi_oversold': 30.0, 'blend_momentum': 0.7}  IS=1.045  OOS=0.264
  W2 2022-06-19->2023-06-19  best={'lookback_months': 15, 'rsi_oversold': 35.0, 'blend_momentum': 0.5}  IS=1.049  OOS=-1.205
  W3 2023-06-19->2024-06-19  best={'lookback_months': 15, 'rsi_oversold': 35.0, 'blend_momentum': 0.5}  IS=0.747  OOS=0.447


In [10]:
oos = wf.oos_metric
verdict = 'DEPLOY CANDIDATE' if oos >= 0.6 else ('MARGINAL -- MONITOR' if oos >= 0.4 else 'NEEDS MORE WORK')
print('=' * 50)
print(f'OOS Sharpe: {oos:.3f}')
print(f'VERDICT: {verdict}')
print('=' * 50)

OOS Sharpe: -0.019
VERDICT: NEEDS MORE WORK


## 8. Honest Verdict

**What worked:** The 60/40 blend smooths the equity curve vs pure momentum.
Mean reversion fires primarily during high-vol episodes where momentum goes flat.
Low return correlation suggests genuine diversification.

**What didn't:** Mean reversion alone has poor CAGR -- it is a complement, not a
standalone strategy. In prolonged drawdowns both signals are off simultaneously.

**Deployment decision:** Only update `strategy_runner.py` if OOS Sharpe >= 0.6 AND
no single window has OOS Sharpe below 0.0.